<a href="https://colab.research.google.com/github/HLZHarry/AlphaLink-Muti-Modal-RAG/blob/main/Step_3_The_Multi_Modal_Parser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install the unstrucutred library
!pip install "unstructured[all-docs]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of unstructured-inference to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.2/538.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.4 MB/s eta 0:00:00
   ━

In [1]:
import os
from google.colab import drive

# 1. Force a clean remount to clear the 'FileNotFound' cache
drive.mount('/content/drive', force_remount=True)

# 2. Re-verify the path exists
# Note: Sometimes 'My Drive' has a space in the actual Linux path
base_path = "/content/drive/MyDrive/AlphaLink-RAG/data/processed_filings"

if os.path.exists(base_path):
    print(f"✅ Success! Path is now visible to Python.")
    print(f"Files found: {len(os.listdir(base_path))}")
else:
    # If it still fails, let's find where 'AlphaLink-RAG' actually is
    print("❌ Path still not found. Searching for AlphaLink-RAG...")
    !find /content/drive/MyDrive -name "AlphaLink-RAG" -type d

Mounted at /content/drive
✅ Success! Path is now visible to Python.
Files found: 442


In [2]:
# Mardown Transformation Script
import os
import spacy
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from unstructured.partition.auto import partition
from google.colab import drive
from tqdm.notebook import tqdm

In [3]:
# 1. MOUNT DRIVE
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive', force_remount=True)

In [4]:
# 2. CONFIGURATION
BASE_DIR = Path("/content/drive/MyDrive/AlphaLink-RAG/data")
INPUT_DIR = BASE_DIR / "processed_filings"
OUTPUT_DIR = BASE_DIR / "markdown_filings"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# 3. GET FILES
all_files = [f for f in INPUT_DIR.iterdir() if f.is_file() and not f.name.startswith('.')]
print(f"🚀 Total files to check: {len(all_files)}")

🚀 Total files to check: 442


In [6]:
# 4. THE ROBUST LOOP
for file_path in tqdm(all_files, desc="Parsing Progress"):
    output_path = OUTPUT_DIR / f"{file_path.stem}.md"

    # CHECKPOINT: Skip if already processed
    if output_path.exists():
      print(f"⏭️ Skipped: {file_path.name}")
      continue


    print(f"⚙️ Parsing: {file_path.name} ({file_path.stat().st_size // 1024} KB)...")

    try:
        # We use 'strategy="fast"' to bypass the heavy NLP models that crash on large files
        # This keeps the headers and tables but runs 10x faster.
        elements = partition(
            filename=str(file_path),
            strategy="fast",
            combine_under_n_chars=0 # Keeps document structure granular
        )

        content = "\n\n".join([str(el) for el in elements])

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(content)

    except Exception as e:
        # FALLBACK: If the parser still chokes, we save the raw text so we don't lose data
        if "exceeds maximum" in str(e) or "E088" in str(e):
            print(f"⚠️ NLP limit hit on {file_path.name}. Falling back to direct extraction.")
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                raw_text = f.read()
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(raw_text)
        else:
            print(f"❌ Error on {file_path.name}: {e}")

Parsing Progress:   0%|          | 0/442 [00:00<?, ?it/s]

⏭️ Skipped: NVDA_10-K_0001045810-26-000021.txt
⏭️ Skipped: NVDA_10-K_0001045810-26-000021.html
⏭️ Skipped: NVDA_10-K_0001045810-25-000023.txt
⏭️ Skipped: NVDA_10-K_0001045810-25-000023.html
⏭️ Skipped: NVDA_10-K_0001045810-24-000029.html
⏭️ Skipped: NVDA_10-K_0001045810-24-000029.txt
⏭️ Skipped: NVDA_10-K_0001045810-23-000017.txt
⏭️ Skipped: NVDA_10-K_0001045810-23-000017.html
⏭️ Skipped: NVDA_10-Q_0001045810-25-000230.txt
⏭️ Skipped: NVDA_10-Q_0001045810-25-000209.txt
⏭️ Skipped: NVDA_10-Q_0001045810-25-000230.html
⏭️ Skipped: NVDA_10-Q_0001045810-25-000209.html
⏭️ Skipped: NVDA_10-Q_0001045810-25-000116.html
⏭️ Skipped: NVDA_10-Q_0001045810-25-000116.txt
⏭️ Skipped: NVDA_10-Q_0001045810-24-000316.txt
⏭️ Skipped: NVDA_10-Q_0001045810-24-000264.txt
⏭️ Skipped: NVDA_10-Q_0001045810-24-000316.html
⏭️ Skipped: NVDA_10-Q_0001045810-23-000227.html
⏭️ Skipped: NVDA_10-Q_0001045810-24-000124.html
⏭️ Skipped: NVDA_10-Q_0001045810-23-000227.txt
⏭️ Skipped: NVDA_10-Q_0001045810-24-000124.txt
⏭️ 